In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

% of women who survived: 0.7420382165605095


In [5]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of men who survived: 0.18890814558058924


In [6]:
faixa_idade = train_data.loc[train_data.Age.between(20, 50), "Survived"]
rate_faixa_idade = sum(faixa_idade) / len(faixa_idade)

print("% of age 20-50 who survived:", rate_faixa_idade)

% of age 20-50 who survived: 0.3888888888888889


In [7]:
from sklearn.ensemble import RandomForestClassifier

# (SibSp + Parch + 1 "o próprio passageiro")
for df in [train_data, test_data]:
  df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

y = train_data["Survived"]

features = ["Pclass", "Sex", "FamilySize", "Age"]
# features = ["Sex", "FamilySize", "Age"]

# 2. Preenche valores faltantes (ex.: mediana de Age calculada no treino)
mediana_idade = train_data["Age"].median()
X_raw = train_data[features].copy()
X_test_raw = test_data[features].copy()

X_raw["Age"] = X_raw["Age"].fillna(mediana_idade)
X_test_raw["Age"] = X_test_raw["Age"].fillna(mediana_idade)

# 3. Converte colunas categóricas (como Sex) para numéricas e alinha as colunas
X = pd.get_dummies(X_raw, dtype=int)
X_test = pd.get_dummies(X_test_raw, dtype=int)
X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

# 4. Treina e prevê
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({
    "PassengerId": test_data["PassengerId"].to_numpy(),
    "Survived": predictions,
})


In [8]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split

# Opção A: validação cruzada (média da acurácia em 5 divisões do treino)
scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print(f"Acurácia média (CV 5-fold): {scores.mean():.4f} ({scores.mean():.2%})")

# Opção B: separando uma fatia de treino e outra de validação
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model.fit(X_tr, y_tr)
val_preds = model.predict(X_val)
print(
    f"Acurácia na validação: {accuracy_score(y_val, val_preds):.4f}"
    f" ({accuracy_score(y_val, val_preds):.2%})"
)

Acurácia média (CV 5-fold): 0.8283 (82.83%)
Acurácia na validação: 0.8268 (82.68%)


In [9]:
output.to_csv("submission.csv", index=False)